# Optional local recovery practice

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_12/materials/notebooks/05_mongodb_logical_recovery_short.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_12/materials/notebooks/05_mongodb_logical_recovery_short.ipynb)

> **Optional legacy reference.** The required single-session Week 12 uses [the Atlas notebook](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_12/materials/notebooks/05_mongodb_atlas_recovery.ipynb). This older notebook adds no submission and must not be presented as evidence of a live Atlas connection.

A restore has the right number of tickets, the right IDs, and real dates. Can it still be wrong?

**Optional practice:** run the supplied setup, inspect a saved data file, repair one wrong subject, and explain the result in **2–3 sentences**. You edit **one line in Cell 06**. The code cells are numbered **01–07** so you can follow the classroom demonstration.

**Environment:** Colab with `mongomock`, a Python imitation of part of MongoDB's API. You need a Google account for Colab; you do **not** need Atlas, a MongoDB server, a URI, or a database account. This exercise demonstrates document recovery.

**Start:** save a copy in Drive. Run the numbered cells from top to bottom. Each run of Cell 02 creates a fresh, isolated practice database. Keep your final outputs and your explanation in your saved copy.

| Minutes | Work |
|---|---|
| 0–3 | Cells 01–02: start the supplied practice environment |
| 3–7 | Cells 03–05: export, restore, and find the missed error |
| 7–11 | Cells 06–07: repair one document and repeat the repair |
| 11–15 | Write 2–3 sentences and save your optional practice |

## 01 — Install the practice packages

Run this supplied cell once. It can take a little time the first time a Colab runtime starts. `pymongo` supplies the BSON-aware JSON tools; `mongomock` supplies the in-memory practice database.

In [ ]:
# Cell 01 — supplied setup; no database credentials needed.
%pip -q install "pymongo>=4.13,<5" "mongomock>=4.3,<5"

## 02 — Start with five fresh tickets

A **document** is one record with named fields. A **collection** holds documents. Here, `tickets` is the collection. `ticket_id` is our application ID; MongoDB also gives each document its own `_id`.

We create two different database names: one **source** and one **restore target**. Only the restore target will be damaged later. The fixture is synthetic: it contains no real service requests.

Run this cell as provided. Expected source count: **5**.

In [ ]:
# Cell 02 — supplied fixture; rerun to start fresh.
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4
import json
import mongomock
from bson import json_util
from bson.json_util import CANONICAL_JSON_OPTIONS

client = mongomock.MongoClient(tz_aware=True)
run_id = uuid4().hex[:8]
SOURCE_DB = f"cst4714_source_{run_id}"
RESTORE_DB = f"cst4714_restore_{run_id}"
assert SOURCE_DB != RESTORE_DB
source_tickets = client[SOURCE_DB]["tickets"]
restore_tickets = client[RESTORE_DB]["tickets"]
JSON_OPTIONS = CANONICAL_JSON_OPTIONS.with_options(tz_aware=True)
practice_files = TemporaryDirectory(prefix="cst4714_recovery_")
EXPORT_FILE = Path(practice_files.name) / "tickets.extended.json"

source_documents = [
    {"ticket_id": 1001, "status": "open", "priority": "high",
     "subject": "Streetlight dark near bus stop",
     "opened_at": datetime(2026, 2, 1, 23, 10, tzinfo=timezone.utc)},
    {"ticket_id": 1002, "status": "in_progress", "priority": "medium",
     "subject": "Missed recycling pickup",
     "opened_at": datetime(2026, 2, 2, 15, 45, tzinfo=timezone.utc)},
    {"ticket_id": 1003, "status": "resolved", "priority": "urgent",
     "subject": "Low water pressure",
     "opened_at": datetime(2026, 2, 3, 12, 5, tzinfo=timezone.utc)},
    {"ticket_id": 1004, "status": "new", "priority": "low",
     "subject": "Broken bench slat",
     "opened_at": datetime(2026, 2, 4, 17, 20, tzinfo=timezone.utc)},
    {"ticket_id": 1005, "status": "resolved", "priority": "high",
     "subject": "Overflowing corner bin",
     "opened_at": datetime(2026, 2, 5, 14, 0, tzinfo=timezone.utc)},
]
source_tickets.create_index("ticket_id", unique=True, name="unique_ticket_id")
source_tickets.insert_many(source_documents)
print("Source and restore target are different:", SOURCE_DB != RESTORE_DB)
print("Source ticket count:", source_tickets.count_documents({}))
for row in source_tickets.find({}).sort("ticket_id", 1):
    print(row["ticket_id"], row["subject"])

## 03 — Save an artifact with date types

An **artifact** is the saved file we will recover from. Ordinary JSON has no native date type. **Extended JSON** uses markers such as `$date` to represent BSON types in text. `json_util.dumps` writes those markers; `json_util.loads` reads them back into the appropriate Python values.

Run this cell. Notice that ordinary `json.loads` sees a **dictionary** where the date marker is, while the BSON-aware reader reconstructs a **datetime**.

This file is in the temporary Colab runtime. It is enough for this exercise, but a real recovery copy must be retained outside the system/runtime you might lose. The source documents stay fixed while the export loop runs.

In [ ]:
# Cell 03 — export source documents; this does not read the restore target.
export_documents = list(source_tickets.find({}).sort("ticket_id", 1))
EXPORT_FILE.write_text(
    json_util.dumps(export_documents, json_options=JSON_OPTIONS, indent=2) + "\n",
    encoding="utf-8",
)
export_text = EXPORT_FILE.read_text(encoding="utf-8")
plain_reader = json.loads(export_text)
bson_reader = json_util.loads(export_text, json_options=JSON_OPTIONS)
print("Saved documents:", len(bson_reader))
print("Extended JSON date marker:", plain_reader[0]["opened_at"])
print("Ordinary JSON reader sees:", type(plain_reader[0]["opened_at"]).__name__)
print("BSON-aware reader sees:", type(bson_reader[0]["opened_at"]).__name__)
print("Recovered date:", bson_reader[0]["opened_at"].isoformat())

## 04 — Restore elsewhere and check the result

**Restore** means loading saved data back into a database. We load the file into the separate restore target. The supplied function `verify_restore()` compares four things with the **saved file**: count, IDs, date types, and complete document values. You do not need to rewrite this function.

All four checks should start **True**. The date check asks only whether each value is a `datetime`; the complete-values check also checks whether it is the correct date.

The source has a unique `ticket_id` index. The restored collection initially has only its automatically created `_id` index. **This document file does not restore index definitions, validators, users, or application configuration.** Rebuilding or testing a validator is **not required today**; we only need to recognize the limit of this recovery.

In [ ]:
# Cell 04 — restore to a separate target, then compare against the file.
saved_documents = json_util.loads(
    EXPORT_FILE.read_text(encoding="utf-8"), json_options=JSON_OPTIONS
)
saved_by_id = {row["ticket_id"]: row for row in saved_documents}
restore_tickets.delete_many({})  # only this run's isolated restore target
restore_tickets.insert_many(saved_documents)

def canonical(rows):
    ordered = sorted(rows, key=lambda row: row["ticket_id"])
    return json_util.dumps(ordered, json_options=JSON_OPTIONS, sort_keys=True)

def verify_restore():
    expected = json_util.loads(
        EXPORT_FILE.read_text(encoding="utf-8"), json_options=JSON_OPTIONS
    )
    actual = list(restore_tickets.find({}).sort("ticket_id", 1))
    checks = {
        "Count matches saved file": len(actual) == len(expected),
        "IDs match saved file": [row["ticket_id"] for row in actual]
            == sorted(row["ticket_id"] for row in expected),
        "Date values are datetime": all(
            isinstance(row.get("opened_at"), datetime) for row in actual
        ),
        "Complete values match saved file": canonical(actual) == canonical(expected),
    }
    for label, passed in checks.items():
        print(f"{label}: {passed}")
    return checks

baseline_checks = verify_restore()
assert all(baseline_checks.values()), "The starting restore must match the file."
print("Source indexes:", sorted(source_tickets.index_information()))
print("Restore indexes:", sorted(restore_tickets.index_information()))

## 05 — A check can pass while a subject is wrong

The supplied incident changes **ticket 1004** in the restore target. It does not change the source or the saved file. Before running, predict which checks will fail.

Expected result: count, IDs, and date-type checks still say **True**. Complete values say **False**. Read the saved and current subjects printed below. A correct count does not prove that the records contain correct information.

This cell intentionally introduces the problem. If you rerun it after repairing the ticket, you will need to repair the ticket again.

In [ ]:
# Cell 05 — intentionally damage one subject in the restore target only.
DAMAGED_TICKET = 1004
restore_tickets.update_one(
    {"ticket_id": DAMAGED_TICKET},
    {"$set": {"subject": "Text replaced by an incorrect migration"}},
)
damaged_checks = verify_restore()
print("Saved subject:", saved_by_id[DAMAGED_TICKET]["subject"])
print("Current subject:", restore_tickets.find_one(
    {"ticket_id": DAMAGED_TICKET}
)["subject"])

## 06 — Your repair: change one line

`saved_by_id` is a Python dictionary containing the documents read from the saved file. Its keys are ticket IDs. `DAMAGED_TICKET` identifies the ticket to recover.

Replace `None` on the `recovery_document` line with the lookup that selects the saved document for `DAMAGED_TICKET`. Use the form `saved_by_id[the_ticket_id]`, putting the supplied variable inside the brackets. Do not type the correct subject by hand: recover its saved document.

Then run this cell. `replace_one` replaces the document that matches the ticket ID. Expected first repair: **Matched: 1**, **Modified: 1**, and all four checks **True**. If you run this cell again immediately, **Modified: 0** is correct because the saved document is already there.

In [ ]:
# Cell 06 — YOUR ONE-LINE CHANGE.
recovery_document = None  # TODO: select the saved document for DAMAGED_TICKET.

if recovery_document is None:
    print("Repair not completed: replace None with the saved-document lookup.")
else:
    assert recovery_document["ticket_id"] == DAMAGED_TICKET
    repair = restore_tickets.replace_one(
        {"ticket_id": DAMAGED_TICKET}, recovery_document
    )
    assert repair.matched_count == 1, "The damaged ticket must already exist."
    print("Matched:", repair.matched_count, "Modified:", repair.modified_count)
repair_checks = verify_restore()

## 07 — Repeat the repair: no sixth ticket

**Idempotent** means repeating an operation has the same final effect as doing it once. After a successful repair, this same replacement should match one document but change zero documents. The collection should still have **5** tickets, and all four checks should stay **True**.

Run this provided cell. Report a missing repair as incomplete.

In [ ]:
# Cell 07 — repeat the same saved-document replacement.
if recovery_document is None:
    print("INCOMPLETE: finish Cell 06 before checking the repeated repair.")
else:
    repeated_repair = restore_tickets.replace_one(
        {"ticket_id": DAMAGED_TICKET}, recovery_document
    )
    print("Repeated repair — matched:", repeated_repair.matched_count,
          "modified:", repeated_repair.modified_count)
    assert repeated_repair.matched_count == 1
    assert repeated_repair.modified_count == 0, "Finish the first repair in Cell 06."
print("Ticket count after repeat:", restore_tickets.count_documents({}))
final_checks = verify_restore()
if recovery_document is not None:
    assert restore_tickets.count_documents({}) == 5
    assert all(final_checks.values())
    print("Document repair verified. Collection rules were not restored by this file.")

## Your explanation — 2–3 sentences

Double-click this text cell and replace the placeholder below. Explain **why the first three checks missed the error**, **how you recovered the document**, and **one item this document file would not restore before an application reopens**.

**My explanation:** [Write your 2–3 sentences here.]

## Save your optional practice

Keep the outputs from Cells 01–07 and your 2–3 sentence explanation in your own saved Drive copy. The failed complete-values check in Cell 05 and repaired results in Cells 06–07 are useful evidence to revisit.

**This legacy notebook has no required submission.** Do not add a Brightspace response or notebook attachment for it. For this week's required work and single Colab-link submission, use the [current Atlas notebook](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_12/materials/notebooks/05_mongodb_atlas_recovery.ipynb).

### Optional longer reference

The original [full logical recovery notebook](05_mongodb_logical_recovery.ipynb) includes additional index reconstruction, an optional Atlas path, validator demonstrations, and tool comparisons. Those extension activities add no required work this week.

For later reading: [MongoDB Extended JSON](https://www.mongodb.com/docs/manual/reference/mongodb-extended-json/) and [mongodump](https://www.mongodb.com/docs/database-tools/mongodump/).
